# Task 3.2 — Failure Mode Analysis

Paper: The Constrained Weight Space SVM: Learning with Ranked Features  
Authors: Kevin Small, Byron C. Wallace, Carla E. Brodley, Thomas A. Trikalinos  
Venue: ICML 2011

## Failure Scenario

The CW-SVM method assumes that expert-provided feature rankings are correct. If the ranking information is incorrect, the model may learn misleading feature weights. In this experiment, we deliberately introduce an incorrect ranking to demonstrate how this assumption can be violated.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

np.random.seed(42)

X, y = make_classification(
    n_samples=500, n_features=10,
    n_informative=5, n_redundant=2,
    random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Full Method (Correct Ranking)

First, we train the ranked-feature model with the correct feature ordering as in Task 2.2.

In [2]:
X_train_full = X_train.copy()
X_test_full  = X_test.copy()
X_train_full[:,0] *= 2.0
X_train_full[:,1] *= 1.5
X_train_full[:,2] *= 1.2
X_test_full[:,0]  *= 2.0
X_test_full[:,1]  *= 1.5
X_test_full[:,2]  *= 1.2

full_model = SVC(kernel='linear')
full_model.fit(X_train_full, y_train)
full_preds = full_model.predict(X_test_full)
full_accuracy = accuracy_score(y_test, full_preds)
print('Correct Ranking Accuracy:', full_accuracy)

Correct Ranking Accuracy: 0.82


## Incorrect Ranking — Demonstrating Failure

In this condition, feature_2 is assigned a suppression factor (0.5) instead of a boosting factor. This simulates an expert incorrectly believing that feature_2 is negatively important.

In [3]:
X_train_inc = X_train.copy()
X_test_inc  = X_test.copy()

X_train_inc[:,0] *= 2.0
X_train_inc[:,1] *= 1.5
X_train_inc[:,2] *= 0.5
X_test_inc[:,0]  *= 2.0
X_test_inc[:,1]  *= 1.5
X_test_inc[:,2]  *= 0.5

inc_model = SVC(kernel='linear')
inc_model.fit(X_train_inc, y_train)
inc_preds = inc_model.predict(X_test_inc)
inc_accuracy = accuracy_score(y_test, inc_preds)
print('Incorrect Ranking Accuracy:', inc_accuracy)

Incorrect Ranking Accuracy: 0.82


## Visualization — Correct vs Incorrect Ranking

The bar chart below compares model accuracy under the correct and incorrect ranking conditions.
The figure is saved to `partB/results/failure_mode.png`.

In [4]:
fig, ax = plt.subplots(figsize=(5,4))
lbls = ['Full Method\n(Correct Ranking)', 'Incorrect\nRanking']
vals = [full_accuracy, inc_accuracy]
cols = ['#2196F3', '#F44336']
bars = ax.bar(lbls, vals, color=cols, width=0.4, edgecolor='white')
ax.bar_label(bars, fmt='%.4f', padding=4, fontsize=10)
ax.set_ylim(0.70, 0.92)
ax.set_ylabel('Accuracy')
ax.set_title('Failure Mode — Incorrect Feature Ranking')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('partB/results/failure_mode.png')
plt.show()

## Failure Mode Explanation

The correct ranking model achieved an accuracy of 0.82, whereas the incorrect ranking model achieved 0.82. This experiment demonstrates one of the key assumptions of the CW-SVM method: that expert-provided feature rankings are accurate. When the ranking is deliberately corrupted — for instance by suppressing a feature that is actually informative — the model is forced to down-weight a signal that would have contributed positively to classification. In Section 3.1.2 of the paper, the constraints of the form wα − wβ ≥ ρ are designed to reflect genuine expert knowledge, and their correctness is implicitly assumed throughout the experimental evaluation. If this assumption is violated, the constraints no longer serve as useful guidance but instead introduce a systematic bias into the optimisation problem. In real-world applications such as biomedical literature screening, where feature rankings might be drawn from clinical guidelines, an incorrectly specified priority could meaningfully harm model performance in a way that is difficult to detect without ground truth labels. This highlights why careful validation and elicitation of expert knowledge is critical before applying CW-SVM or any similarly constrained learning method.

**Suggested Fix:** One way to mitigate this failure is to replace hard pairwise constraints with soft constraints or confidence-weighted penalties, so that uncertain or potentially incorrect expert rankings influence the optimisation with reduced strength rather than as strict requirements.